# Fase 2 — prompt, rota e callbacks

Demonstração local dos componentes usados pelo mesmo agente do menu 9. As entradas são **fixtures locais**; nenhuma chamada à OpenAI ou às APIs de negócio ocorre neste notebook. O teste real do agente é feito no terminal.


In [ ]:
from pathlib import Path
import sys
raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/secretaria').is_dir())
sys.path.insert(0, str(raiz / 'src'))
from secretaria.agente_agenda import montar_prompt_agenda
from secretaria.roteamento import roteador_pedido
from secretaria.observabilidade import RastroAgente


## Prompt PTCF e exemplos
A estrutura delimita papel, tarefa, regras e formato. Os exemplos são instruções de comportamento; não são respostas medidas de um modelo.


In [ ]:
prompt = montar_prompt_agenda('ana')
assert all(marca in prompt for marca in ('[PESSOA]', '[TAREFA]', '[CONTEXTO E REGRAS]', '[EXEMPLOS]', '[FORMATO]'))
print('Seções:', [linha for linha in prompt.splitlines() if linha.startswith('[')])


## Roteamento dinâmico leve
A rota orienta o agente; não confirma visitas nem restringe ferramentas. Os serviços Python continuam validando efeitos e identidade.


In [ ]:
casos = ['Agende uma visita ao cliente Alfa', 'O CNPJ está cadastrado?', 'Olá']
rotas = [roteador_pedido.invoke({'pergunta': pergunta, 'anteriores': []})['rota'] for pergunta in casos]
assert rotas == ['visita', 'cliente', 'geral']
print(list(zip(casos, rotas)))


## Callback sem conteúdo privado
Eventos abaixo são acionados **manualmente** para verificar o formato do rastro. Eles não demonstram uma chamada real ao agente.


In [ ]:
from uuid import uuid4
rastro = RastroAgente()
id_execucao = uuid4()
rastro.on_chat_model_start({}, [], run_id=id_execucao)
rastro.on_tool_start({'name': 'buscar_cliente'}, 'entrada omitida', run_id=id_execucao)
rastro.on_tool_end('retorno omitido', run_id=id_execucao)
rastro.on_llm_end(None, run_id=id_execucao)
assert [e['evento'] for e in rastro.eventos] == ['modelo_iniciado', 'ferramenta_iniciada', 'ferramenta_concluida', 'modelo_concluido']
print(rastro.eventos)
